# Finance and Graphics Demo
## Calculate Bollinger Bands
## Graph with MatplotLib 
## Use a widget to make it interactive
## Trading Strategy!



In [ ]:

import pandas as pd
import matplotlib.pyplot as plt 

import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

In [ ]:
try:
    import yfinance as yf
except: 
    !pip install yfinance
    import yfinance as yf 


## Let's download a stocks Dataset


In [ ]:
# Define the tickers for multiple stocks
tickers = ["AAPL", "GOOG", "AMZN", "META", "NVDA", "MSFT"]

# Download historical data for the past 5 years
stock_data = yf.download(tickers, start="2019-01-01", end="2026-02-19", group_by="ticker")

# Save the data to a CSV file (optional)
stock_data.to_csv("stock_data.csv")

# Display the first few rows
stock_data

In [ ]:
# Extract AAPL data
ap_prices = stock_data['AAPL'].copy()
ap_prices

# Bollinger Bands Explained

Bollinger Bands are a **technical analysis tool** used to measure market volatility and identify overbought or oversold conditions. They consist of three lines:

1. **Middle Band** – A simple moving average (SMA), typically a **20-day SMA**.
2. **Upper Band** – The SMA plus **2 standard deviations**.
3. **Lower Band** – The SMA minus **2 standard deviations**.

Since standard deviation measures volatility, the bands expand when the market is volatile and contract when the market is stable.

---

## **Formula for Bollinger Bands**
Given a time period \( N \) (typically 20 days):

- **Middle Band (SMA)**:  
  $$ \text{SMA} = \frac{1}{N} \sum_{i=1}^{N} P_i $$  
  where \( P_i \) is the price at day \( i \).

- **Upper Band**:  
  $$ \text{Upper Band} = \text{SMA} + (k \times \sigma) $$

- **Lower Band**:  
  $$ \text{Lower Band} = \text{SMA} - (k \times \sigma) $$
where:
-  $\sigma $ is the standard deviation of prices over $ N $ days.
- $k$ is typically **2**, meaning bands are **2 standard deviations** from the SMA.

---

## **How to Interpret Bollinger Bands**
1. **Price Near the Upper Band → Overbought**
   - If the price touches or moves above the upper band, the asset **may be overbought**, signaling a possible reversal downward.

2. **Price Near the Lower Band → Oversold**
   - If the price touches or moves below the lower band, the asset **may be oversold**, signaling a possible upward reversal.

3. **Bollinger Band Squeeze → Low Volatility**
   - When bands contract, it signals **low volatility** and often precedes a breakout in either direction.

4. **Bollinger Band Expansion → High Volatility**
   - When bands widen, volatility is increasing. This often happens after a strong move in price.

---


## Calculate the new Bollinger Bands columns
`Pandas` has a `.rolling` argument that we can pass in

In [ ]:
window = 20  

In [ ]:
ap_prices['SMA'] = ap_prices['Close'].rolling(window).mean()  # Simple Moving Average
ap_prices['StdDev'] = ap_prices['Close'].rolling(window).std()  # Standard Deviation
ap_prices['Upper'] = ap_prices['SMA'] + (2 * ap_prices['StdDev'])  # Upper Band
ap_prices['Lower'] = ap_prices['SMA'] - (2 * ap_prices['StdDev'])  # Lower Band

In [ ]:
ap_prices = ap_prices.dropna()
ap_prices

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ap_prices.index, ap_prices['Close'], label="Closing Price", color='blue')
plt.plot(ap_prices.index, ap_prices['SMA'], label="SMA", color='black', linestyle="dashed")
plt.plot(ap_prices.index, ap_prices['Upper'], label="Upper Band", color='red')
plt.plot(ap_prices.index, ap_prices['Lower'], label="Lower Band", color='green')
plt.fill_between(ap_prices.index, ap_prices['Upper'], ap_prices['Lower'], color='gray', alpha=0.2)
plt.title("Bollinger Bands (Apple 5 years)")
plt.legend()
plt.show()

In [ ]:
# Filter the data to only include the last year
last_year = ap_prices.loc[ap_prices.index >= ap_prices.index.max() - pd.DateOffset(years=1)].copy()


In [ ]:
# Plot Bollinger Bands for the last year
plt.figure(figsize=(12, 6))
plt.plot(last_year.index, last_year['Close'], label="Closing Price", color='blue')
plt.plot(last_year.index, last_year['SMA'], label="SMA", color='black', linestyle="dashed")
plt.plot(last_year.index, last_year['Upper'], label="Upper Band", color='red')
plt.plot(last_year.index, last_year['Lower'], label="Lower Band", color='green')

# Fill the area between the upper and lower bands
plt.fill_between(last_year.index, last_year['Upper'].values, last_year['Lower'].values, color='gray', alpha=0.2)

plt.title("Bollinger Bands (Last Year)")
plt.legend()
plt.show()

## Identify overbought conditions where the closing price is above the upper Bollinger Band

In [ ]:
overbought = last_year[last_year['Close'] > last_year['Upper']]

overbought

# add oversold
oversold = last_year[last_year['Close'] < last_year['Lower']]

oversold

### Plot Bollinger Bands with overbought conditions highlighted


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(last_year.index, last_year['Close'], label="Closing Price", color='blue')
plt.plot(last_year.index, last_year['SMA'], label="SMA", color='black', linestyle="dashed")
plt.plot(last_year.index, last_year['Upper'], label="Upper Band", color='red')
plt.plot(last_year.index, last_year['Lower'], label="Lower Band", color='green')

# Fill the Bollinger Bands range
plt.fill_between(last_year.index, last_year['Upper'].values, last_year['Lower'].values, color='gray', alpha=0.2)

# Highlight overbought points
plt.scatter(overbought.index, overbought['Close'], color='red', label="Overbought", marker='o')

# Highlight oversold points
plt.scatter(oversold.index, oversold['Close'], color='green', label="Oversold", marker='o')

plt.title("Bollinger Bands - Overbought & Oversold Periods (Last Year)")
plt.legend()
plt.show()

## Lets build some widgets!

In [ ]:
tickers

In [ ]:
# Create dropdown for stock selection
stock_picker = widgets.Dropdown(
    options=tickers,
    value="AAPL",
    description="Stock:"
)

# Create slider for selecting the time window in years
time_window = widgets.IntSlider(
    value=1,  # Default to 1 year
    min=1,
    max=5,
    step=1,
    description="Years:"
)



# Create slider for MA window size
ma_window = widgets.IntSlider(
    value=20,  # Default to 20 days
    min=5,
    max=50,
    step=5,
    description="MA Window:"
)

In [ ]:
# Display widgets
display(stock_picker, time_window, ma_window)

## Lets make the Graph again but lets let it update with the widgets

- put the whole graph in a function 
- fuction takes 3 arguments which are the widgets
- plot it and have it update as widgets change


In [ ]:
# Function to compute and plot Bollinger Bands
def plot_bollinger_bands(stock, years, window=20):
    # Extract only the selected stock's data
    df = stock_data[stock].copy()
    
    # Get the last date in the dataset and compute the start date
    end_date = df.index.max()
    start_date = end_date - pd.DateOffset(years=years)
    
    # Filter data based on the selected time window
    df = df.loc[start_date:end_date]

    # Compute Bollinger Bands
    df['SMA'] = df['Close'].rolling(window).mean()
    df['StdDev'] = df['Close'].rolling(window).std()
    df['Upper'] = df['SMA'] + (2 * df['StdDev'])
    df['Lower'] = df['SMA'] - (2 * df['StdDev'])

    # Drop NaN values
    df = df.dropna()

    # Plot Bollinger Bands
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['Close'], label="Closing Price", color='blue')
    plt.plot(df.index, df['SMA'], label="SMA", color='black', linestyle="dashed")
    plt.plot(df.index, df['Upper'], label="Upper Band", color='red')
    plt.plot(df.index, df['Lower'], label="Lower Band", color='green')
    plt.fill_between(df.index, df['Upper'], df['Lower'], color='gray', alpha=0.2)

    plt.title(f"Bollinger Bands for {stock} ({years} Year(s), MA={window})")
    plt.legend()
    plt.show()



In [ ]:
# Create an interactive widget with output
output = widgets.Output()

# Function to update the output widget
def update_plot(stock, years, window):
    with output:
        output.clear_output(wait=True)
        plot_bollinger_bands(stock, years, window)

# Create interactive widgets
interactive_plot = widgets.interactive(update_plot, stock=stock_picker, years=time_window, window=ma_window)

# Display widgets and output
display(stock_picker, time_window, ma_window, output)

# Call the function once to display initial plot
update_plot(stock_picker.value, time_window.value, ma_window.value)

## Can we make a trading rule 
- buy when oversold for 2 days 
- sell when overbought 2 days

## When would trades get made?

In [ ]:
# Create signals for oversold and overbought conditions
last_year['oversold_signal'] = (last_year['Close'] < last_year['Lower']).astype(int)
last_year['overbought_signal'] = (last_year['Close'] > last_year['Upper']).astype(int)

# Check for 2 consecutive days of oversold/overbought
# Rolling sum of 2 periods - if it equals 2, then we had 2 consecutive days
last_year['oversold_2days'] = last_year['oversold_signal'].rolling(2).sum() == 2
last_year['overbought_2days'] = last_year['overbought_signal'].rolling(2).sum() == 2

last_year[['Close', 'Lower', 'Upper', 'oversold_signal', 'overbought_signal', 'oversold_2days', 'overbought_2days']].tail(20)

In [ ]:
# Generate trading signals
# Buy signal: When we hit 2 consecutive oversold days (day 2 of the signal)
# Sell signal: When we hit 2 consecutive overbought days (day 2 of the signal)

# Shift to ensure we're only counting the transition into the 2-day period
last_year['buy_signal'] = last_year['oversold_2days'] & ~last_year['oversold_2days'].shift(1, fill_value=False)
last_year['sell_signal'] = last_year['overbought_2days'] & ~last_year['overbought_2days'].shift(1, fill_value=False)

# Get the dates when trades would be made
buy_dates = last_year[last_year['buy_signal']]
sell_dates = last_year[last_year['sell_signal']]

print(f"Number of BUY signals: {len(buy_dates)}")
print(f"Number of SELL signals: {len(sell_dates)}")
print("\nBuy signals:")
print(buy_dates[['Close', 'Lower', 'Upper']])
print("\nSell signals:")
print(sell_dates[['Close', 'Lower', 'Upper']])

In [ ]:
# Visualize the trading signals on the Bollinger Bands chart
plt.figure(figsize=(14, 7))

# Plot Bollinger Bands
plt.plot(last_year.index, last_year['Close'], label="Closing Price", color='blue', linewidth=2)
plt.plot(last_year.index, last_year['SMA'], label="SMA", color='black', linestyle="dashed")
plt.plot(last_year.index, last_year['Upper'], label="Upper Band", color='red', alpha=0.7)
plt.plot(last_year.index, last_year['Lower'], label="Lower Band", color='green', alpha=0.7)
plt.fill_between(last_year.index, last_year['Upper'].values, last_year['Lower'].values, color='gray', alpha=0.2)

# Mark BUY signals (green triangles pointing up)
if len(buy_dates) > 0:
    plt.scatter(buy_dates.index, buy_dates['Close'], color='darkgreen', marker='^', 
                s=200, label="BUY Signal", zorder=5, edgecolors='black', linewidth=1.5)

# Mark SELL signals (red triangles pointing down)
if len(sell_dates) > 0:
    plt.scatter(sell_dates.index, sell_dates['Close'], color='darkred', marker='v', 
                s=200, label="SELL Signal", zorder=5, edgecolors='black', linewidth=1.5)

plt.title("Bollinger Bands Trading Strategy: Buy after 2 Days Oversold, Sell after 2 Days Overbought", 
          fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Price ($)", fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Trading Strategy Performance

Let's calculate a simple returns analysis:
- Assume we buy at the close price on a buy signal
- Assume we sell at the close price on a sell signal
- Calculate the return for each buy-sell pair

In [ ]:
# Simple trading performance calculation
# Match each buy with the next sell signal

if len(buy_dates) > 0 and len(sell_dates) > 0:
    trades = []
    
    for buy_date in buy_dates.index:
        # Find the next sell signal after this buy
        future_sells = sell_dates[sell_dates.index > buy_date]
        
        if len(future_sells) > 0:
            sell_date = future_sells.index[0]
            buy_price = buy_dates.loc[buy_date, 'Close']
            sell_price = sell_dates.loc[sell_date, 'Close']
            
            profit = sell_price - buy_price
            return_pct = (profit / buy_price) * 100
            
            trades.append({
                'Buy Date': buy_date,
                'Buy Price': buy_price,
                'Sell Date': sell_date,
                'Sell Price': sell_price,
                'Profit ($)': profit,
                'Return (%)': return_pct
            })
    
    trades_df = pd.DataFrame(trades)
    print(f"\nTotal number of completed trades: {len(trades_df)}")
    print(f"Average return per trade: {trades_df['Return (%)'].mean():.2f}%")
    print(f"Total profit: ${trades_df['Profit ($)'].sum():.2f}\n")
    print(trades_df)
else:
    print("Not enough buy/sell signals to calculate performance")

### Compare to Buy-and-Hold Strategy

Let's compare the trading strategy to simply buying at the beginning and holding until the end.

In [ ]:
# Calculate Buy-and-Hold performance for the same period
buy_hold_start_price = last_year['Close'].iloc[0]
buy_hold_end_price = last_year['Close'].iloc[-1]
buy_hold_profit = buy_hold_end_price - buy_hold_start_price
buy_hold_return = (buy_hold_profit / buy_hold_start_price) * 100

print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\n📊 BUY-AND-HOLD STRATEGY:")
print(f"   Start Date: {last_year.index[0].strftime('%Y-%m-%d')}")
print(f"   End Date: {last_year.index[-1].strftime('%Y-%m-%d')}")
print(f"   Buy Price: ${buy_hold_start_price:.2f}")
print(f"   End Price: ${buy_hold_end_price:.2f}")
print(f"   Total Return: {buy_hold_return:.2f}%")
print(f"   Total Profit: ${buy_hold_profit:.2f}")

if len(buy_dates) > 0 and len(sell_dates) > 0 and len(trades_df) > 0:
    trading_total_return = trades_df['Return (%)'].sum()
    trading_total_profit = trades_df['Profit ($)'].sum()
    
    print(f"\n📈 BOLLINGER TRADING STRATEGY:")
    print(f"   Number of Trades: {len(trades_df)}")
    print(f"   Average Return per Trade: {trades_df['Return (%)'].mean():.2f}%")
    print(f"   Total Return: {trading_total_return:.2f}%")
    print(f"   Total Profit: ${trading_total_profit:.2f}")
    
    print(f"\n{'='*60}")
    print("COMPARISON:")
    print(f"{'='*60}")
    
    if trading_total_return > buy_hold_return:
        outperformance = trading_total_return - buy_hold_return
        print(f"✅ Trading Strategy OUTPERFORMED by {outperformance:.2f}%")
    else:
        underperformance = buy_hold_return - trading_total_return
        print(f"❌ Trading Strategy UNDERPERFORMED by {underperformance:.2f}%")
    
    profit_diff = trading_total_profit - buy_hold_profit
    if profit_diff > 0:
        print(f"   Extra profit: ${profit_diff:.2f}")
    else:
        print(f"   Less profit: ${abs(profit_diff):.2f}")
else:
    print("\n⚠️  Not enough trading signals to compare strategies")

In [ ]:
# Visualize the comparison
if len(buy_dates) > 0 and len(sell_dates) > 0 and len(trades_df) > 0:
    strategies = ['Buy & Hold', 'Bollinger Trading']
    returns = [buy_hold_return, trades_df['Return (%)'].sum()]
    colors = ['skyblue', 'lightcoral']
    
    # Determine which is better and adjust colors
    if returns[1] > returns[0]:
        colors[1] = 'lightgreen'
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(strategies, returns, color=colors, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, ret in zip(bars, returns):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{ret:.2f}%',
                ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylabel('Total Return (%)', fontsize=12)
    ax.set_title('Strategy Performance Comparison (Last Year)', fontsize=14, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()